# Lets use this as our spot to make random stuff for the project so our main notebook is not clouded by other stuff

In [3]:
### Function for christoffels
def calcChristoffel(g , indices):

    #This program will calculate the connection coefficients for g using the Levi-Civita connection

    #Packages
    import sympy as sp

    #Find g inverse
    ginv = g.inv()

    #Initialize the connections as a 4x4x4 array
    christoffel = sp.MutableDenseNDimArray.zeros(4,4,4)

    #The Levi-Civita Connection is
    #Γ^m_np = .5 * g^ms ( deriv_p(g_sn) + deriv_n(g_sp) - deriv_s(g_np) )
    for mu in range(0 , 4):
        for nu in range(0 , 4):
            for rho in range(0 , 4):
                expr = 0
                for sigma in range(0 , 4):
                    expr += .5 * ginv[mu , sigma] * (
                        sp.diff(g[sigma , nu] , indices[rho]) +
                        sp.diff(g[sigma , rho] , indices[nu]) -
                        sp.diff(g[nu , rho] , indices[sigma])
                    )

                christoffel[mu , nu , rho] = expr


    return (christoffel , ginv) #Return ginv also just incase its useful elsewhere






In [5]:
#Testing christoffel maker
import sympy as sp
t , r , theta , phi= sp.symbols('t r theta phi')
M = 5
g = sp.Matrix([[-(1 - 2*M/r) , 0 , 0 , 0] , [0 , 1/(1 - 2*M/r) , 0, 0] , [0 , 0 , r**2 , 0] , [0 , 0 , 0 , r**2 * sp.sin(theta)**2]])

indices = {0:t , 1:r , 2:theta , 3:phi}

calcChristoffel(g , indices)





Given metric was Matrix([[-1 + 10/r, 0, 0, 0], [0, 1/(1 - 10/r), 0, 0], [0, 0, r**2, 0], [0, 0, 0, r**2*sin(theta)**2]])
Inverse metric is Matrix([[-r/(r - 10), 0, 0, 0], [0, (r - 10)/r, 0, 0], [0, 0, r**(-2), 0], [0, 0, 0, 1/(r**2*sin(theta)**2)]])
Christoffel symbols are [[[0, 5.0/(r*(r - 10)), 0, 0], [5.0/(r*(r - 10)), 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]], [[5.0*(r - 10)/r**3, 0, 0, 0], [0, -5.0*(r - 10)/(r**3*(1 - 10/r)**2), 0, 0], [0, 0, 10.0 - 1.0*r, 0], [0, 0, 0, -1.0*(r - 10)*sin(theta)**2]], [[0, 0, 0, 0], [0, 0, 1.0/r, 0], [0, 1.0/r, 0, 0], [0, 0, 0, -1.0*sin(theta)*cos(theta)]], [[0, 0, 0, 0], [0, 0, 0, 1.0/r], [0, 0, 0, 1.0*cos(theta)/sin(theta)], [0, 1.0/r, 1.0*cos(theta)/sin(theta), 0]]]


([[[0, 5.0/(r*(r - 10)), 0, 0], [5.0/(r*(r - 10)), 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]], [[5.0*(r - 10)/r**3, 0, 0, 0], [0, -5.0*(r - 10)/(r**3*(1 - 10/r)**2), 0, 0], [0, 0, 10.0 - 1.0*r, 0], [0, 0, 0, -1.0*(r - 10)*sin(theta)**2]], [[0, 0, 0, 0], [0, 0, 1.0/r, 0], [0, 1.0/r, 0, 0], [0, 0, 0, -1.0*sin(theta)*cos(theta)]], [[0, 0, 0, 0], [0, 0, 0, 1.0/r], [0, 0, 0, 1.0*cos(theta)/sin(theta)], [0, 1.0/r, 1.0*cos(theta)/sin(theta), 0]]],
 Matrix([
 [-r/(r - 10),          0,       0,                      0],
 [          0, (r - 10)/r,       0,                      0],
 [          0,          0, r**(-2),                      0],
 [          0,          0,       0, 1/(r**2*sin(theta)**2)]]))

-sqrt(r*(r**3 - 10.0*r**2 + 101.0*r - 1000.0))/(10.0*r - 100.0)


In [15]:
def findUtInit(is_null , ri , vi , g , indices):

    #Import packages
    import sympy as sp
    import numpy as np

    #For our integration to be correct, we need to find the t-component of the initial four velocity.
    #This ut determines the evolution of the system as a massive or massless particle.
    #Per the diff geo, the magnitude of the four velo is found via u^m*u^n*g_mn
    #This magnitude must equal -1 for a massive particle and 0 for a massless particle

    if is_null:
        condition = 0

    else:
        condition = -1

    #Lets sub in the initial position symbols into the metric
    subs_dict = {
        indices[0] : ri[0] ,
        indices[1] : ri[1] ,
        indices[2] : ri[2] ,
        indices[3] : ri[3] ,
    }

    g_initial = g.subs(subs_dict)

    #Lets initialize our unknown ut
    ut = sp.symbols("ut" , positive = True)
    ui = [ut , vi[0] , vi[1] , vi[2]]

    #Now we do the inner product correctly
    expr = 0
    for m in range(0 , 4):
        for n in range(0 , 4):
            expr += ui[m] * ui[n] * g_initial[m , n]

    #Make the expression an equation using condition
    inner_product_constraint = sp.Eq(expr , condition)

    #Solve for ut
    time_part = sp.solve(inner_product_constraint , ut)

    #Put the four velocity together
    initial_4velo = [time_part , vi[0] , vi[1] , vi[2]]
    return(initial_4velo)



print(findUtInit(is_null = False , ri = [0 , 50 , sp.pi/2 , 0] , vi = [.1 , .1 , .1] , g = g , indices = indices ))

[7.98533812684222]


# Slowly putting pieces together here

In [21]:
##### USER CONTROLLED INPUTS #####

### Note: Inputs for a Schwartzchild metric with mass M = 5 in Areal coordinates given as default. If the user wants to change the inputs, follow the patterns given :)

#Please define the symbols of your coordinate system here, make your timelike coordinate the FIRST symbol!
coords = sp.symbols('t r theta phi')

#Please define your metric here, make each column refer to above symbols in the same order!
g = sp.Matrix(
    [[-(1 - (10 / r)) , 0 , 0 , 0] , #TIMELIKE COLUMN
     [0 , 1/(1 - (10 / r)) , 0 , 0] , #SPACELIKE COLUMN
     [0 , 0 , r**2 , 0] , #SPACELIKE COLUMN
     [0 , 0 , 0 , r**2 * sp.sin(theta)**2]] #SPACELIKE COLUMN
)

#Please input your initial 4-position, make each entry refer to coordinates in same order as above!
ri = [0 , 100 , sp.pi/2 , 0]

#Please input your initial 3-velocity (ie the spatial velocities, not the timelike one), follow same spatial order as above
vi = [50 , 0 , 0]

#Please input the amount of "code time" you would like this simulation to go by and the step amount
dlam = .005 #Step size
lamTot = 5000 #Total sim code time

#Please input if this particle is massless or not
massless_particle = False

##### END OF USER CONTROLLED INPUTS #####


##### FUNCTIONS FOR CODE #####

#This function will calculate the inverse metric and corresponding Christoffel symbols
def calcChristoffel(g , indices):

    #This program will calculate the connection coefficients for g using the Levi-Civita connection

    #Packages
    import sympy as sp

    #Find g inverse
    ginv = g.inv()

    #Initialize the connections as a 4x4x4 array
    christoffel = sp.MutableDenseNDimArray.zeros(4,4,4)

    #The Levi-Civita Connection is
    #Γ^m_np = .5 * g^ms ( deriv_p(g_sn) + deriv_n(g_sp) - deriv_s(g_np) )
    for mu in range(0 , 4):
        for nu in range(0 , 4):
            for rho in range(0 , 4):
                expr = 0
                for sigma in range(0 , 4):
                    expr += .5 * ginv[mu , sigma] * (
                        sp.diff(g[sigma , nu] , indices[rho]) +
                        sp.diff(g[sigma , rho] , indices[nu]) -
                        sp.diff(g[nu , rho] , indices[sigma])
                    )

                christoffel[mu , nu , rho] = expr

    #We have the christoffels yay

    return (christoffel , ginv) #Return ginv also just incase its useful elsewhere

#This function will calculate the
def findUtInit(is_null , ri , vi , g , indices):

    #Import packages
    import sympy as sp
    import numpy as np

    #For our integration to be correct, we need to find the t-component of the initial four velocity.
    #This ut determines the evolution of the system as a massive or massless particle.
    #Per the diff geo, the magnitude of the four velo is found via u^m*u^n*g_mn
    #This magnitude must equal -1 for a massive particle and 0 for a massless particle

    if is_null:
        condition = 0

    else:
        condition = -1

    #Lets sub in the initial position symbols into the metric
    subs_dict = {
        indices[0] : ri[0] ,
        indices[1] : ri[1] ,
        indices[2] : ri[2] ,
        indices[3] : ri[3] ,
    }

    g_initial = g.subs(subs_dict)

    #Lets initialize our unknown ut
    ut = sp.symbols("ut" , positive = True)
    ui = [ut , vi[0] , vi[1] , vi[2]]

    #Now we do the inner product correctly
    expr = 0
    for m in range(0 , 4):
        for n in range(0 , 4):
            expr += ui[m] * ui[n] * g_initial[m , n]

    #Make the expression an equation using condition
    inner_product_constraint = sp.Eq(expr , condition)

    #Solve for ut
    time_part = sp.solve(inner_product_constraint , ut)
    return(time_part)


##### END OF FUNCTIONS FOR CODE #####


##### RUN SIM #####

#Create an indices dictionary
indices = {}
for i in range(0 , 4):
    indices[i] = coords[i]

#Calculate inverse metric and christoffel symbols
(Connections , ginv) = calcChristoffel(g = g , indices = indices)

#Calculate the initial timelike component of the 4-velocity
ui = findUtInit(is_null = massless_particle, ri = ri , vi = vi , g = g , indices = indices)










